# BERT 기본개념2


## 파인튜닝(전이학습)

똑똑한 BERT(사전학습된) 모델을 가져다가 전이학습을 통해 우리가 원하는 목적으로 쓸 수 있다.  (파인튜닝 곧 전이학습이다)

**전이학습(Transfer Learning): 사전학습(Pre-training)된 BERT를 가져와, 맞춤형 데이터로 간단한 파인튜닝(파라미터 재조정)만으로 학습**

> 파라미터가 많이 변하지 않는 정교하고 세밀한 조정이 이루진다. 그래서 파인 튜닝임!

사전학습 BERT를 데려와서 특수한 목적(Task)에 맞게 살짝만 교육시킬 수 있다. 대표적으로 4가지 Task가 있다. 

1. 한 문장의 성격 분류 (Sentiment Analysis)
   > [CLS] 토큰 위치 위에 Dense 층을 하나 얹어서 긍정/부정을 판단한다. 
   > DENSE 층은 뭘까? -> DENSE 층은 BERT가 추출한 복잡한 문맥 정보를 우리가 이해할 수 있도록 긍정 부정! 이런 식으로 압축하는 최종 의사결정 기구이다. CLS 토큰은 BERT base 모델 기준 768차원 벡터일 것이다. 여기서 DENSE층이 768차원의 벡터를 가중치 고하고.. 다 더하고 이런 과정을 통해 분류할 클래스 숫자만큼으로 뽑아낸다. 

즉, DENSE 층을 추가하는 것이 우리가 파인튜닝을 한 것이다. 파인튜닝을 통해 DENSE 층을 추가해서 이게 긍정이야 부정이야? 라고 묻는 새로운 세포층을 하나 덧붙이는 것이다. 파인튜닝을 할 때 아래쪽 BERT의 기존 파라미터는 최대한 유지하면서(물론, 수정하기는 하지) DENSE 층의 가중치를 집중적으로 학습시킨다
-> 그 결과 CLS 벡터가 들어오면 긍정 부정 나눌 수 있다 

2. 단어에 대한 태깅 (NER)
    > 각 단어 위치마다 정답(사람, 조직 등)을 매긴다. 
    > 여기에도 DENSE 층은 똑같은 원리이다. 단, 감정분류처럼 CLS 토큰만 보는 것이 아니라 모든 토큰 위에 전용 DENSE 층을 올린다. 각 단어의 벡터를 각각의 층에 넣는다. 이때 DENSE 층은 역시 압축기이자 분류기 역할을 한다(층 내의 가중치와 768차원 벡터를 곱한다. 파인튜닝 과정에서 이 단어가 사람 이름일 때 어떤 특징을 나타내고, 조직 이름일 때 어떤 특징을 나타내고.. 이런 것들이 학습되어 있다)

3. 문장 간 관계 분석 (Natural Language Inference)
    > 앞서 배운 [SEP]로 구분된 두 덩어리의 관계가 참(True)인지 거짓(False)인지 분류한다.
    > nsp 연습했으니까 이거 가져다 쓰자!즉, 앞 문장과 뒷 문장간의 관계를 추론할 수 있다. 인과관계야 and관계야 관계 없는거야 이런 식으로 

4. 새로운 내용 생성: 질의 응답 (QA)
 > 본문 속에서 정답이 시작되는 위치와 끝나는 위치를 찾아낸다. 


결국, 4가지 Task(파인튜닝을 통한)에서 Dense층은 BERT가 문맥을 반영하여 새롭게 만들어낸 벡터를 우리가 이해할 수 있는 구체적인 정답 양식으로 바꿔주는 최종의사결정 기구이다. 
> 모든 Dense 층은 BERT라는 거대한 엔진이 뽑아준 '원재료(768차원 벡터)'를 가져와서, 우리가 풀고자 하는 **구체적인 문제의 정답 형태(Label)**로 가공해주는 **'마지막 변환기'**!!


즉, 파인튜닝 과정에서 Dense 층은 가장 핵심적인 학습 대상이다. 사전 학습된 BERT 모델에는 Dense 층이 없는데, 우리가 특정 문제를 풀기 위해 새로 이어붙인 층이다!!


## SBERT

BERT는 문맥 안에서 단어가 무슨 뜻인지 본다면, Sentence BERT는 문장과 문장이 얼마나 비슷하지? -> 즉, 문장 수준의 비교를 한다. 


BERT는 단어 수준의 이해이다. 

*다시 BERT를 생각해 보자면, 각 토큰이 BERT layer를 지나면 사전적 의미만 있던 메마른 단어들이 풍부한 맥락을 반영한 벡터가 된다. BERT layer를 통과하면서 (self -attention 과정을 거침) 문맥 반영한 벡터가 된다. 최종 레이어 통과한 벡터는 단순한 임베딩이 아니다. 사과를 먹었다와 사과를 건넸다 의 사과 라는 단어의 벡터가 다르게 임베딩된다.*

BERT는 각 단어마다 벡터 1개가 나온다.(768차원의 벡터로) 즉, 이것은 단어(토큰) 별 벡터이기 때문에 문장 전체를 대표하는 벡터가 아니다. 

그럼 BERT는 두 문장간의 관계를 어떻게 보냐?
> [CLS] 문장 A [SEP] 문장 B [SEP] 을 통째로 넣는다. 이러면 문장 A의 모든 단어가 문장 B의 모든 단어를 self attention으로 일일이 훑어야 한다. 전부 다 계산하니 정확도는 높을 수 있어도 시간이 엄청 많이 든다.


이때 등장한 것이 SBERT이다. SBERT는 문장 수준의 비교를 한다

SBERT는 각 문장을 미리 하나의 **'고유한 요약 벡터**로 만들어 보관하는 전략을 쓴다. 
> 여기서 풀링(pooling)이 등장한다.   
> BERT 레이어를 통과한 512개의 단어 벡터들을 평균 내거나(Mean Pooling) 가장 강한 신호만 추출하여, 문장 전체를 대표하는 단 하나의 768차원 벡터로 압축한다. 

문장 비교 방식은?  

문장 A를 넣어서 그 문장을 가장 잘 대표하는 벡터로(풀링 과정을 거쳐) 추출해서 보관하고, 문장 B에 대해서도 대표 벡터를 추출해서 보관한다. 그 후 두 벡터 사이의 거리(코사인 유사도)만 계산하면 압도적으로 빠르다!

물론, SBERT에서도 Self attention 일어난다. 풀링 전까지는 BERT와 똑같이 일한다. 하지만, 일반 BERT는 문장 A에서 attention 하고 문장 B에서 attention된 토큰들과 다시 attention을 한다. -> 여기서 느려진다!

하지만 SBERT는 대표하는 벡터로 압축하고 두 문장간의 관계는 코사인 유사도로 구한다.
>  일반 BERT는 문장 A와 문장 B의 벡터(토큰들)를 한데 섞어서 Attention 연산을 함
> SBERT: "독립적 요약 후 수학적 비교" - 두 문장을 비교하는데 attention 안하고 코사인 유사도를 할 수 있다!


SBERT는 학습할 때는 BERT처럼 문장 사이의 관계를 배우지만, 실제 사용할 때는 **"야, 매번 둘이 Attention 시키니까 너무 느려! 그냥 각자 자기 문장 안에서만 Attention 해서 요약본(벡터)만 만들어와!"** 

*일반 BERT가 NSP 학습 할 때 A와 B 문장 넣고 attention을 시키면서 학습을 한다!(물론, MLM 에서도 attention으로 학습함)*

결국 BERT의 모든 능력도 attention에서 나오는구나..

정리하자면,

SEBRT: 문장 전체 -> 벡터 1개로 요약(요약 방식은 풀링을 통해 단어벡터를 요약한다)

각 문장을 독립적으로 인코딩 후 코사인 유사도로 즉시 비교한다. 벡터 연산만으로 문장 관계 비교 가능하다. 
> 이 벡터가 임베딩된 위치는 각 토큰이 아닌 문장의 위치 문장의 의미를 반영한 임베딩일 것이다. 그럼 이 벡터끼리 코사인 유사도를 측정하면 문장끼리 유사도를 만들 수 있다는 것이다.

> SBERT로 pooling 거쳐서 문장을 최종 의미있는 벡터를 만들어버리면, 단어가 아닌 문장 간의 의미 유사도를 잡아낼 수 있다. (기사 100건을 각 기사마다 대표 벡터로 만들고, 그 벡터를 이용해서 그럼 유사한 기사끼리 k means로 클러스터링 하는 것도 가능할 것이다. > 즉, 신문기사를 대표하는 벡터로 만들어서 임베딩하고 이를 바탕으로 k-means 클러스터링 하면 유사한 기사끼리 그룹화가 된다)


*근데 cls 토큰도 문장을 대표하는 벡터 아니였나..?*
> SBERT는 다른 방식으로 문장 대표하는 벡터를 만든 것이다.
> CLS 에는 의미관계 잘 녹아든 것이고, SBERT에서는 의미의 위치 찾기, 의미 유사도 비교하기에 더 좋은 것으로 이해하자. 
>
> 
> CLS도 문장 전체의 맥락을 머금은 대표 벡터이다. 그런데, BERT의 CLS는 분류를 위한 요약본이다. 하지만 SBERT의 풀링 벡터는 이 문장의 평균적인 의미는 무엇인가? 를 담는 것이 목적이다. CLS는 비교에 적합한 친구가 아니다. SBERT는 파인튜닝을 할 때부터 비슷한 의미 문장은 가까운 위치에, 다른 의미는 멀리 떠러지도록 학습했다.   
>  CLS 토큰 하나 쓰는 것보다 모든 단어의 평균을 내는 풀링 방식이 더 효과적이었더라! 실험해보니까!


## 요즘 트렌드

GPT가 너무 성능이 좋다.   
하지만, 의미기준으로 분류하는 자동화 시스템으 하겠다? -> SBERT 여전히 사용한다.

의미분류 기초로 하는 자동화 시스템은 SBERT가 지피티 시대에도 사용한다. 

지피티가 감정분석 1 ~5 점 매기는 것 잘한다. 근데 BERT 계열 사용할 떄가 있다.

무료이고, GPU 크게 필요 없음 -> 내 작업이 매일매일 실시간으로 해야하는 작업이라면? 조금 덜 정확하더라도 BERT 계열을 하는 것이 더 좋다. 


성능은 요즘 GPT 보다 떨어지긴 하지만, 이런 경우 SBERT를 아직도 사용한다! GPT의 시대에서도!!

> 허깅페이스 안에 BERT 엄청 많다. 그 중 좋은거 찾아서 좋은 시스템을 만들 수도 있다!!

### NOTE

대규모 텍스트를 매일 처리해야 하는 환경에서 BERT 계열을 고려할 필요가 있다.

SBERT는 자동화 시스템에서 매우 좋은 성능 발휘한다. 

벡터 데이터베이스화: SBERT는 문장을 고정된 크기의 숫자 뭉치(벡터)로 바꿉니다. 사용자님이 모은 카페 게시글 10만 건을 미리 벡터로 바꿔두면, 새로운 글이 왔을 때 **수학적 계산(코사인 유사도)**만으로 0.001초 만에 유사한 글을 찾습니다.

GPT의 한계: GPT API를 써서 10만 건을 매번 비교하려면 토큰 비용이 감당 안 될 정도로 발생하고, 네트워크 응답을 기다리는 데 한세월이 걸립니다.

결론: 의미 기반의 '실시간 분류/중복 체크/추천' 자동화 시스템을 구축할 때는 SBERT가 훨씬 경제적이고 빠릅니다.

**GPT가 잘하는데 왜 BERT로 감정 분석을 할까?**  
GPT는 '범용적'으로 잘하지만, BERT는 **'특정 도메인'**에 맞게 길들일 수 있기 때문입니다.

미세 조정(Fine-tuning)의 힘: GPT는 "효과 최고!"를 단순 긍정으로 보겠지만, 사용자님이 **BERT를 특정 데이터를 통해 훈련(Fine-tuning)**시키면 "이건 전형적인 과장 광고 패턴이네?"라며 더 정교하게 잡아낼 수 있습니다.

매일 쏟아지는 수만 건의 커뮤니티 데이터를 실시간으로 크롤링해서 분석해야 한다면, API를 거치지 않고 내 컴퓨터(로컬)에서 즉시 처리하는 BERT가 속도 면에서 압승입니다.

| 용어 | 설명 | 비유 |
| :--- | :--- | :--- |
| Self-Attention | 한 문장 내 모든 단어 쌍의 관련성 계산 | 학생이 자기 노트의 모든 내용을 상호 참조 |
| Multi-Head | 여러 관점에서 동시에 어텐션 계산 | 8명이 같은 글을 각자 다른 관점에서 분석 |
| Q, K, V | 질문(Query), 열쇠(Key), 값(Value) | 도서관에서 질문으로 책(Key)을 찾아 내용(V) 확인 |
| Positional Encoding | 단어의 순서 정보를 임베딩에 추가 | 책의 페이지 번호를 내용에 표시 |
| Add & Norm | 잔차 연결 + 레이어 정규화 | 새 내용을 배울 때 기존 지식도 함께 유지 |
| Masked Attention | 미래 단어를 보지 못하게 마스킹 | 시험 중 다음 문제 미리보기 방지 |
| MLM | 빈칸 채우기 학습 (BERT) | 빈칸 뚫린 문장 완성하기 시험 |
| NSP | 다음 문장 예측 학습 (BERT) | 두 문장이 이어지는지 판단하는 독해 문제 |
| Fine-tuning | 사전학습 모델을 특정 과제에 맞게 조정 | 일반 교육 받은 학생이 전공 과목 심화 학습 |